In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

cols = ['sentiment', 'id', 'date', 'query', 'user', 'text']
df = pd.read_csv('data/raw/sentiment140.csv', encoding='latin-1', names=cols)
df['sentiment'] = df['sentiment'].map({0: 'negative', 4: 'positive'})
df_sample = df.sample(n=10000, random_state=42)
df_sample['text_length'] = df_sample['text'].apply(len)
df_sample['word_count'] = df_sample['text'].apply(lambda x: len(str(x).split()))
print("Data loaded:", df_sample.shape)

Data loaded: (10000, 8)


In [9]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_tweet(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens 
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df_sample['clean_text'] = df_sample['text'].apply(clean_tweet)
print("Cleaning done. Sample:")
print(df_sample[['text', 'clean_text']].head(3))

Cleaning done. Sample:
                                                     text  \
541200             @chrishasboobs AHHH I HOPE YOUR OK!!!    
750     @misstoriblack cool , i have no tweet apps  fo...   
766711  @TiannaChaos i know  just family drama. its la...   

                                               clean_text  
541200                                          ahhh hope  
750                                  cool tweet apps razr  
766711  know family drama lamehey next time hang kim g...  


In [10]:
empty = df_sample['clean_text'].apply(lambda x: len(str(x).strip()) == 0).sum()
print(f"Empty rows after cleaning: {empty}")

df_sample['clean_word_count'] = df_sample['clean_text'].apply(lambda x: len(str(x).split()))
print(f"Avg word count before cleaning: {df_sample['word_count'].mean():.1f}")
print(f"Avg word count after cleaning: {df_sample['clean_word_count'].mean():.1f}")

df_sample.to_csv('data/processed/nike_clean.csv', index=False)
print("Saved to data/processed/nike_clean.csv")

Empty rows after cleaning: 59
Avg word count before cleaning: 13.2
Avg word count after cleaning: 6.7
Saved to data/processed/nike_clean.csv


In [11]:
# Drop empty rows
df_clean = df_sample[df_sample['clean_text'].apply(lambda x: len(str(x).strip()) > 0)].copy()
df_clean = df_clean.reset_index(drop=True)

print(f"Original sample: 10,000")
print(f"After removing empty rows: {len(df_clean)}")
print(f"Removed: {10000 - len(df_clean)} rows")

# Save final clean dataset
df_clean.to_csv('data/processed/nike_clean_final.csv', index=False)
print("\n[OK] Final clean dataset saved to data/processed/nike_clean_final.csv")
print("\nSample of clean data:")
print(df_clean[['sentiment', 'text', 'clean_text']].head(5).to_string())

Original sample: 10,000
After removing empty rows: 9941
Removed: 59 rows

[OK] Final clean dataset saved to data/processed/nike_clean_final.csv

Sample of clean data:
  sentiment                                                                                                                                       text                                                                         clean_text
0  negative                                                                                                     @chrishasboobs AHHH I HOPE YOUR OK!!!                                                                           ahhh hope
1  negative                                                                                  @misstoriblack cool , i have no tweet apps  for my razr 2                                                               cool tweet apps razr
2  negative  @TiannaChaos i know  just family drama. its lame.hey next time u hang out with kim n u guys like have a sleepover or w